# 15.2 - Explainable AI: Opening the Black Box

**Module 15 - Ethics & Responsible AI** | Netsetos GenAI Engineering

This notebook turns a single opaque loan-model score into a defensible reason, one method per cell: exact feature attribution, LIME local surrogates, exact SHAP over all coalitions, global-vs-local importance, minimal counterfactuals, and a fidelity check that separates a real explanation from a confident alibi. Everything is keyless arithmetic on one illustrative logistic loan model - no `shap`/`lime` library, no API call.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

A single reproducibility note - the exercises below use fixed, seeded values so every run prints identical numbers. No packages are installed and nothing is imported, because every cell is plain Python arithmetic on one hard-coded loan model.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

This is a comment-only cell that documents the reproducibility contract for the notebook: the numbers are deterministic by construction, not sampled at runtime.

**How the code works - step by step**
- **The comment** - flags that the lesson uses seeded/fixed values throughout, so outputs are stable across runs.
- **No imports, no installs** - the only libraries used later are `itertools` and `math`, imported inline in the SHAP cell.

**In one line:** a note, not code - every result below is deterministic.

**What you'll see:** (no output - environment prep)

## 1 - The black box: a score with no reason

A production model is millions of weights collapsed into one number, and that number is enough to *decide* but not to *justify*. This cell runs a tiny logistic loan model, prints the opaque score and DENY verdict, and states the gap that the rest of the notebook fills - because under GDPR Art. 22 and the EU AI Act a high-risk decision with no reason is not just unsatisfying, it can be unlawful.

In [ ]:
# THE BLACK BOX: a model returns a DECISION and a score, but not a REASON. For a loan denial that is not just unsatisfying -
# under GDPR Art. 22 and the EU AI Act it can be illegal. Explainability turns one opaque number into an accountable story.
def loan_model(applicant):
    # a real model is millions of weights; the OUTPUT is a single score, and the score alone hides the "why"
    w = {"income": 0.9, "debt_ratio": -1.6, "credit_age": 0.4, "recent_defaults": -1.1}
    z = sum(w[f] * applicant[f] for f in w)      # standardized features, illustrative weights
    score = 1 / (1 + 2.71828 ** (-z))            # logistic squash to 0..1
    return round(score, 3), "APPROVE" if score >= 0.5 else "DENY"
applicant = {"income": 0.4, "debt_ratio": 0.8, "credit_age": 0.2, "recent_defaults": 0.6}
score, decision = loan_model(applicant)
print("model output:  score = {}   decision = {}".format(score, decision))
print("the applicant asks: WHY was I denied? The score {} answers nothing.".format(score))
print("A decision without a reason is not accountable - and for a high-risk system it is not lawful. That is the gap XAI fills.")

# Output:
# model output:  score = 0.182   decision = DENY
# the applicant asks: WHY was I denied? The score 0.182 answers nothing.
# A decision without a reason is not accountable - and for a high-risk system it is not lawful. That is the gap XAI fills.

**Explanation**

The whole cell is a demonstration of the problem, not a solution: it builds a real (if tiny) model, runs it on one applicant, and shows that the raw output answers a decision but explains nothing. Read it as the motivating gap - one number, no 'why'.

**How the code works - step by step**
- **`loan_model`** - a weighted sum of four standardized features (income, debt_ratio, credit_age, recent_defaults) squashed through a hand-written logistic function into a 0..1 score, returning the score plus APPROVE/DENY at a 0.5 threshold.
- **`applicant`** - one applicant's feature vector.
- **The prints** - emit the score and decision, then pose the applicant's question ('WHY?') and state that the bare score is not accountable and, for a high-risk system, not lawful.

**In one line:** run the model, show it emits a verdict with no reason - that is the gap XAI fills.

**What you'll see:** The model prints `score = 0.182   decision = DENY`, then two lines noting the score answers nothing and that a reasonless high-risk decision is not accountable or lawful.

## 2 - Feature attribution: the exact decomposition

When the model is **additive**, the reason is exactly true, not approximated: each feature's contribution is its weight times its value, and the contributions sum to the score with nothing left over. This cell computes and ranks those signed contributions - the gold-standard explanation, available whenever the model is linear in its features.

In [ ]:
# FEATURE ATTRIBUTION: for an ADDITIVE model the reason decomposes exactly - each feature's contribution is its weight times
# its value, and the contributions sum to the score. This is the simplest, most faithful explanation when the model allows it.
w = {"income": 0.9, "debt_ratio": -1.6, "credit_age": 0.4, "recent_defaults": -1.1}
applicant = {"income": 0.4, "debt_ratio": 0.8, "credit_age": 0.2, "recent_defaults": 0.6}
contrib = {f: round(w[f] * applicant[f], 3) for f in w}
z = round(sum(contrib.values()), 3)
print("per-feature contribution to the log-odds (weight x value):")
for f, c in sorted(contrib.items(), key=lambda kv: kv[1]):
    bar = ("-" if c < 0 else "+") * max(1, int(abs(c) * 20))
    print("  {:<16} {:+.3f}  {}".format(f, c, bar))
print()
print("total log-odds z = {}  ->  the two NEGATIVE drivers are debt_ratio ({}) and recent_defaults ({}).".format(
      z, contrib["debt_ratio"], contrib["recent_defaults"]))
print("Now the denial has a reason: high debt ratio and recent defaults outweighed income. That is an explanation you can act on.")

# Output:
# per-feature contribution to the log-odds (weight x value):
#   debt_ratio       -1.280  -------------------------
#   recent_defaults  -0.660  -------------
#   credit_age       +0.080  +
#   income           +0.360  +++++++
#
# total log-odds z = -1.5  ->  the two NEGATIVE drivers are debt_ratio (-1.28) and recent_defaults (-0.66).
# Now the denial has a reason: high debt ratio and recent defaults outweighed income. That is an explanation you can act on.

**Explanation**

This cell is the exact decomposition of the same log-odds the previous cell squashed: no surrogate, no perturbation, no approximation error - the sum of the parts IS the model. It shows why glass-box additive models are favoured for high-stakes decisions: their explanation is free and faithful by construction.

**How the code works - step by step**
- **`w` and `applicant`** - the same weights and features as Step 1.
- **`contrib`** - `weight x value` for each feature, its exact signed contribution to the log-odds.
- **`z`** - the sum of contributions, which equals the model's pre-squash log-odds exactly.
- **The sorted loop** - prints each contribution smallest-first with an ASCII bar sized to its magnitude, surfacing the negative drivers.

**In one line:** contribution = weight x value; the parts sum exactly to the score, so the denial gets a reason.

**What you'll see:** A ranked bar chart of contributions - debt_ratio -1.280, recent_defaults -0.660, credit_age +0.080, income +0.360 - summing to `z = -1.5`, with a closing line naming high debt ratio and recent defaults as the reason for denial.

## 3 - LIME: a local linear surrogate

Real models are non-linear - features interact and there is no single global weight for 'income' - so LIME explains *one prediction*: perturb the input around that instance, watch the output move, and read the local slope. This cell probes the neighborhood by finite differences and shows the same feature having opposite local weights at two different applicants - the essential warning that a local weight is not a global truth.

In [ ]:
# LIME: real models are NOT additive, so you approximate the model LOCALLY. Perturb one prediction, see how the output moves,
# and fit a simple linear model in that neighborhood. The local sensitivities are the explanation FOR THIS INSTANCE (not globally).
def model(x):  # a NON-linear model: income and debt interact (their product matters), so no single global weight exists
    return round(0.6 * x["income"] - 1.5 * x["debt_ratio"] - 0.9 * x["income"] * x["debt_ratio"], 3)
base = {"income": 0.4, "debt_ratio": 0.8}
f0 = model(base)
eps = 0.01
local = {}
for feat in base:
    up = dict(base); up[feat] += eps          # probe the neighborhood: nudge one feature, measure the output change
    local[feat] = round((model(up) - f0) / eps, 3)   # local slope = LIME-style local weight
print("prediction at this point: {}".format(f0))
print("local sensitivity (how the output moves as each feature nudges up, near THIS applicant):")
for feat, slope in local.items():
    print("  {:<12} {:+.3f}".format(feat, slope))
far = {"income": 0.9, "debt_ratio": 0.1}
far_slope = round((model({"income": 0.91, "debt_ratio": 0.1}) - model(far)) / eps, 3)
print()
print("income's local weight here is {} but at a different applicant (low debt) it is {} - the SAME feature, different local".format(local["income"], far_slope))
print("effect, because the model is non-linear. LIME explains ONE prediction; do not read a local weight as a global truth.")

# Output:
# prediction at this point: -1.248
# local sensitivity (how the output moves as each feature nudges up, near THIS applicant):
#   income       -0.100
#   debt_ratio   -1.900
#
# income's local weight here is -0.1 but at a different applicant (low debt) it is 0.5 - the SAME feature, different local
# effect, because the model is non-linear. LIME explains ONE prediction; do not read a local weight as a global truth.

**Explanation**

This cell swaps the additive model for a non-linear one (income and debt multiply), then approximates it locally by nudging each feature and measuring the output change - the local-slope idea behind LIME. Its point is the disagreement: probe two different points and the same feature's weight flips sign.

**How the code works - step by step**
- **`model`** - a non-linear model with an `income * debt_ratio` interaction term, so no single global weight exists.
- **`base`, `f0`** - the applicant being explained and the model's prediction there.
- **The `for feat` loop** - nudges each feature up by `eps` and computes `(model(up) - f0) / eps`, the finite-difference local slope (LIME's local weight).
- **`far`, `far_slope`** - repeats the probe at a second, low-debt applicant to show income's local weight change sign.

**In one line:** fit a tangent to the model at one point; move the point and the tangent tilts differently - so never read a local weight globally.

**What you'll see:** Prints the prediction `-1.248` and local sensitivities (income -0.100, debt_ratio -1.900), then notes income's local weight is -0.1 here but +0.5 at a low-debt applicant - the same feature, a different local effect.

## 4 - SHAP: the fair, exact Shapley split

SHAP treats features as players in a game whose payoff is the prediction, and gives each feature its average marginal contribution across every order of adding features - the unique attribution satisfying the Shapley fairness axioms (Shapley 1953, Lundberg-Lee 2017). This cell computes SHAP *exactly* over all coalitions with `itertools` (no `shap` library) and verifies the efficiency property: the values sum to prediction minus baseline.

In [ ]:
# SHAP: the game-theoretic answer. A feature's SHAP value is its average marginal contribution over EVERY order of adding
# features - the unique attribution that is fair (Shapley 1953). Exact here for 3 features via all coalitions; it needs no scipy.
from itertools import combinations
from math import factorial
FEATURES = ["income", "debt_ratio", "credit_age"]
v = {(): 0.10, ("income",): 0.30, ("debt_ratio",): 0.25, ("credit_age",): 0.15,
     ("income", "debt_ratio"): 0.60, ("income", "credit_age"): 0.40, ("debt_ratio", "credit_age"): 0.35,
     ("income", "debt_ratio", "credit_age"): 0.80}   # v(S) = model output with only S present (rest at baseline); income+debt synergise
def val(S): return v[tuple(f for f in FEATURES if f in S)]
n = len(FEATURES)
shap = {}
for f in FEATURES:
    others = [x for x in FEATURES if x != f]
    phi = 0.0
    for k in range(len(others) + 1):
        for S in combinations(others, k):
            weight = factorial(len(S)) * factorial(n - len(S) - 1) / factorial(n)
            phi += weight * (val(set(S) | {f}) - val(set(S)))   # marginal contribution of f given coalition S
    shap[f] = round(phi, 3)
print("SHAP value per feature (average marginal contribution over all orderings):")
for f in FEATURES:
    print("  {:<12} {:+.3f}".format(f, shap[f]))
total, base, pred = round(sum(shap.values()), 3), v[()], v[("income", "debt_ratio", "credit_age")]
print()
print("sum of SHAP values = {}  ==  prediction {} - baseline {} = {}   (the EFFICIENCY property holds exactly).".format(
      total, pred, base, round(pred - base, 3)))
print("SHAP handles the income+debt interaction fairly - it splits the synergy between them, which naive weight x value cannot.")

# Output:
# SHAP value per feature (average marginal contribution over all orderings):
#   income       +0.317
#   debt_ratio   +0.267
#   credit_age   +0.117
#
# sum of SHAP values = 0.701  ==  prediction 0.8 - baseline 0.1 = 0.7   (the EFFICIENCY property holds exactly).
# SHAP handles the income+debt interaction fairly - it splits the synergy between them, which naive weight x value cannot.

**Explanation**

This is the game-theoretic attribution computed from first principles: it enumerates every coalition of features, weights each feature's marginal contribution by the Shapley coefficient, and sums. It handles interactions correctly - the income+debt synergy gets split fairly - which naive weight-times-value cannot, and it self-checks against the efficiency identity.

**How the code works - step by step**
- **`FEATURES`, `v`** - three features and the coalition value function `v(S)`: the model's output with only the features in `S` present (rest at baseline 0.10); income+debt jointly exceed their solo sum, i.e. they synergise.
- **`val`** - looks up `v(S)` by canonicalizing the coalition to feature order.
- **The nested loop** - for each feature, over every coalition `S` of the others, adds `weight * (val(S+f) - val(S))`, where `weight = |S|!(n-|S|-1)!/n!` is the Shapley coefficient.
- **The efficiency check** - compares the SHAP sum against `prediction - baseline`.

**In one line:** average each feature's marginal contribution over all coalitions -> the fair split that sums to prediction minus baseline.

**What you'll see:** Prints SHAP values income +0.317, debt_ratio +0.267, credit_age +0.117, then confirms their sum 0.701 equals prediction 0.8 minus baseline 0.1 (efficiency holds), with a note that the income+debt interaction was split fairly.

## 5 - Global vs local: the forest and the tree

A SHAP value explains one prediction (local); the mean **absolute** SHAP value across many predictions gives global importance - what the model relies on overall. This cell aggregates per-instance SHAP into a global ranking and then surfaces a deliberate disagreement: the globally-least-important feature is the top local driver for one applicant, proving you must report both.

In [ ]:
# GLOBAL vs LOCAL: a SHAP value explains ONE prediction (local). Average the ABSOLUTE SHAP values across many predictions and
# you get GLOBAL feature importance - which feature matters most overall. The global ranking can differ from any single local one.
instances = {   # per-instance SHAP values for three applicants (from the local explainer, illustrative)
    "applicant_1": {"income": 0.32, "debt_ratio": -0.40, "credit_age": 0.10},
    "applicant_2": {"income": 0.05, "debt_ratio": -0.15, "credit_age": 0.42},   # the outlier: credit_age drove this one
    "applicant_3": {"income": 0.28, "debt_ratio": -0.35, "credit_age": 0.08}}
features = ["income", "debt_ratio", "credit_age"]
global_imp = {f: round(sum(abs(inst[f]) for inst in instances.values()) / len(instances), 3) for f in features}
print("GLOBAL importance (mean |SHAP| across applicants):")
for f, imp in sorted(global_imp.items(), key=lambda kv: -kv[1]):
    print("  {:<12} {:.3f}".format(f, imp))
print()
top_local = max(instances["applicant_2"], key=lambda f: abs(instances["applicant_2"][f]))
top_global = max(global_imp, key=global_imp.get)
print("Globally the biggest driver is {}. But for applicant_2 the biggest local driver is {} (SHAP {:+.2f}).".format(
      top_global, top_local, instances["applicant_2"][top_local]))
print("Global tells you what the model relies on overall; local tells you why THIS person got THIS decision. You need both.")

# Output:
# GLOBAL importance (mean |SHAP| across applicants):
#   debt_ratio   0.300
#   income       0.217
#   credit_age   0.200
#
# Globally the biggest driver is debt_ratio. But for applicant_2 the biggest local driver is credit_age (SHAP +0.42).
# Global tells you what the model relies on overall; local tells you why THIS person got THIS decision. You need both.

**Explanation**

This cell is an aggregation-and-contrast harness, not a new explainer: it takes precomputed per-instance SHAP values, averages their magnitudes into global importance, then finds the one applicant whose top local driver contradicts the global ranking. The lesson is that global and local answer different questions and neither substitutes for the other.

**How the code works - step by step**
- **`instances`** - per-instance SHAP values for three applicants; applicant_2 is the planted outlier whose decision credit_age drove.
- **`global_imp`** - mean of `abs(SHAP)` per feature across applicants = global feature importance.
- **The sorted print** - ranks global importance, putting debt_ratio first.
- **`top_local` / `top_global`** - picks applicant_2's largest-magnitude local feature and the top global feature, then contrasts them.

**In one line:** mean |SHAP| -> global importance; one instance can rank features differently, so report both.

**What you'll see:** Prints the global ranking (debt_ratio 0.300, income 0.217, credit_age 0.200), then notes that although debt_ratio leads globally, applicant_2's biggest local driver is credit_age (+0.42) - no contradiction, just different questions.

## 6 - Counterfactuals: the smallest change that flips the decision

An attribution says *why*; the applicant usually wants *what would I have to change?* A counterfactual (Wachter, Mittelstadt & Russell 2017) is the smallest input change that flips the outcome. This cell computes, for each feature, the change that closes the gap to the approval threshold, and picks the smallest actionable one - the concrete, checkable form regulators favour.

In [ ]:
# COUNTERFACTUAL EXPLANATIONS: the most actionable form - "what is the SMALLEST change that would flip the decision?" It answers
# the applicant's real question ("what do I need to do?") and is the form regulators favour, because it is concrete and checkable.
w = {"income": 0.9, "debt_ratio": -1.6, "credit_age": 0.4, "recent_defaults": -1.1}
applicant = {"income": 0.4, "debt_ratio": 0.8, "credit_age": 0.2, "recent_defaults": 0.6}
z = sum(w[f] * applicant[f] for f in w)          # current log-odds (score below 0.5 means DENY: z < 0)
print("current log-odds z = {:.3f}  ->  DENY (needs z >= 0 to flip to APPROVE).".format(z))
gap = 0 - z                                       # how much log-odds must increase to cross the threshold
print("to flip the decision, the log-odds must rise by {:.3f}. Cheapest single-feature counterfactuals:".format(gap))
for f in ["income", "debt_ratio", "recent_defaults"]:
    delta = gap / w[f]                            # change in this feature needed to close the gap (respect the sign)
    direction = "increase" if delta > 0 else "reduce"
    print("  {} {:<16} by {:.2f}  (standardized)".format(direction, f, abs(delta)))
print()
print("The smallest actionable change wins: reducing debt_ratio by {:.2f} flips APPROVE. A counterfactual is an explanation the".format(abs(gap / w["debt_ratio"])))
print("applicant can act on and a regulator can verify - unlike a raw attribution, it says exactly what would have changed the outcome.")

# Output:
# current log-odds z = -1.500  ->  DENY (needs z >= 0 to flip to APPROVE).
# to flip the decision, the log-odds must rise by 1.500. Cheapest single-feature counterfactuals:
#   increase income           by 1.67  (standardized)
#   reduce debt_ratio       by 0.94  (standardized)
#   reduce recent_defaults  by 1.36  (standardized)
#
# The smallest actionable change wins: reducing debt_ratio by 0.94 flips APPROVE. A counterfactual is an explanation the
# applicant can act on and a regulator can verify - unlike a raw attribution, it says exactly what would have changed the outcome.

**Explanation**

This cell reuses the additive log-odds model and inverts it: instead of reading the score, it solves for how far each feature must move to push the log-odds across zero. Each single-feature counterfactual is `gap / weight`, respecting the sign, and the smallest actionable move is the answer the applicant can act on.

**How the code works - step by step**
- **`w`, `applicant`, `z`** - the additive weights, the applicant, and the current log-odds (negative -> DENY).
- **`gap`** - `0 - z`, how much the log-odds must rise to reach the APPROVE threshold.
- **The `for f` loop** - for each candidate feature computes `delta = gap / w[f]`, the standardized change that closes the gap, and labels it increase/reduce by the sign.
- **The closing prints** - highlight the smallest actionable counterfactual (reduce debt_ratio) as an explanation both actionable and verifiable.

**In one line:** solve `weight x delta = gap` per feature; the smallest actionable delta is the counterfactual.

**What you'll see:** Prints the current log-odds `-1.500 -> DENY`, the required rise of 1.500, and three single-feature counterfactuals (increase income 1.67, reduce debt_ratio 0.94, reduce recent_defaults 1.36), concluding that reducing debt_ratio by 0.94 flips to APPROVE.

## 7 - Faithfulness: does the explanation actually match the model?

This is the step that separates explainability from theater. Every method so far produces an explanation, but an explanation is only trustworthy if it *matches the model* - a surrogate can be perfectly plausible and completely wrong. This cell measures **fidelity** (R-squared of the surrogate against the real model on held-out perturbations) for a faithful and an unfaithful surrogate, and shows why a low-fidelity explanation is worse than none.

In [ ]:
# FAITHFULNESS: an explanation can be PLAUSIBLE and still WRONG. A surrogate that does not track the real model is a confident
# alibi, not a reason. Measure FIDELITY (how well the explanation predicts the model on held-out perturbations) before you trust it.
model_outputs   = [0.81, 0.62, 0.44, 0.90, 0.55, 0.33, 0.70, 0.48]   # the real model on 8 perturbations
good_surrogate  = [0.79, 0.64, 0.42, 0.88, 0.57, 0.35, 0.68, 0.50]   # a faithful local linear surrogate
bad_surrogate   = [0.60, 0.58, 0.61, 0.59, 0.62, 0.57, 0.60, 0.58]   # plausible-looking but flat: ignores the real drivers
def r2(y, yhat):
    mean = sum(y) / len(y)
    ss_res = sum((a - b) ** 2 for a, b in zip(y, yhat))
    ss_tot = sum((a - mean) ** 2 for a in y)
    return round(1 - ss_res / ss_tot, 3)
print("fidelity (R^2 of the surrogate against the real model on held-out perturbations):")
print("  faithful surrogate:  R^2 = {}   -> trust its explanation".format(r2(model_outputs, good_surrogate)))
print("  unfaithful surrogate: R^2 = {}  -> reject it: plausible story, wrong reason".format(r2(model_outputs, bad_surrogate)))
print()
print("Always report fidelity with an explanation. A low-fidelity explanation (and, in LLMs, an unfaithful chain-of-thought that")
print("rationalises rather than reveals) is worse than none - it manufactures false confidence. The EU AI Act (Art. 13, 86) requires")
print("meaningful explanations for high-risk decisions, so fidelity is not optional - it is the difference between XAI and theater.")

# Output:
# fidelity (R^2 of the surrogate against the real model on held-out perturbations):
#   faithful surrogate:  R^2 = 0.988   -> trust its explanation
#   unfaithful surrogate: R^2 = 0.025  -> reject it: plausible story, wrong reason
#
# Always report fidelity with an explanation. A low-fidelity explanation (and, in LLMs, an unfaithful chain-of-thought that
# rationalises rather than reveals) is worse than none - it manufactures false confidence. The EU AI Act (Art. 13, 86) requires
# meaningful explanations for high-risk decisions, so fidelity is not optional - it is the difference between XAI and theater.

**Explanation**

This is a measurement harness, not another explainer: it scores two surrogate explanations against the real model's outputs and lets the numbers decide which to trust. The unfaithful surrogate is deliberately plausible-but-flat, and its near-zero fidelity is the whole point - the same failure mode as an LLM's unfaithful chain-of-thought.

**How the code works - step by step**
- **`model_outputs`** - the real model's output on 8 held-out perturbations (the ground truth).
- **`good_surrogate`** - a faithful local linear surrogate that tracks the model closely.
- **`bad_surrogate`** - a plausible-looking but nearly flat surrogate that ignores the real drivers.
- **`r2`** - hand-written R-squared: `1 - ss_res/ss_tot`, how well the surrogate predicts the model.
- **The prints** - report both fidelities and the verdict: trust 0.988, reject 0.025, and always report fidelity.

**In one line:** score each surrogate's R-squared against the model - high fidelity is corroborated, low fidelity is a confident alibi.

**What you'll see:** Prints the faithful surrogate at `R^2 = 0.988` (trust it) and the unfaithful one at `R^2 = 0.025` (reject it - plausible story, wrong reason), closing with the rule that fidelity is mandatory for high-risk explanations under the EU AI Act.

You now have the whole XAI toolkit as running code: decompose an additive score (Step 2), approximate locally with LIME (Step 3), split fairly with exact SHAP and verify efficiency (Step 4), separate global importance from a single local reason (Step 5), give an actionable counterfactual (Step 6), and - above all - measure fidelity so a plausible-but-wrong explanation gets rejected (Step 7). Two questions to ask of any explanation you ship: does it answer what the person actually asked, and have you measured that it is faithful to the model? Lesson 15.6 covers the law that requires these explanations for high-risk systems, and 15.7 folds them into governance.